Note : Use the drive link for the processed dataset

In [2]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

# Paths to the real and fake video directories
real_video_dir = 'D:\major project\dataset\celeb df2\Celeb-real'
fake_video_dir = 'D:\major project\dataset\celeb df2\Celeb-synthesis'

# Video dimensions (adjust as needed)
height, width = 120, 160

# Data lists
X_data = []
y_data = []

# Sample frames every n frames
sample_every = 5

# Preprocess real videos
for video_file in os.listdir(real_video_dir):
    video_path = os.path.join(real_video_dir, video_file)
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Sample every nth frame
        if frame_count % sample_every == 0:
            frame = cv2.resize(frame, (width, height))
            X_data.append(frame)
            y_data.append(0)  # 0 for real videos
        
        frame_count += 1
    
    cap.release()

# Preprocess fake videos
for video_file in os.listdir(fake_video_dir):
    video_path = os.path.join(fake_video_dir, video_file)
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Sample every nth frame
        if frame_count % sample_every == 0:
            frame = cv2.resize(frame, (width, height))
            X_data.append(frame)
            y_data.append(1)  # 1 for fake videos
        
        frame_count += 1
    
    cap.release()

# Convert data to numpy arrays
X_data = np.array(X_data)
y_data = np.array(y_data)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

# Save the preprocessed data for later use
np.save('X_train.npy', X_train)
np.save('X_test.npy', X_test)
np.save('y_train.npy', y_train)
np.save('y_test.npy', y_test)


In [1]:
!pip3 install face_recognition

In [1]:
#THis code is to check if the video is corrupted or not..
#If the video is corrupted delete the video.
import glob
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Dataset
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import face_recognition
#Check if the file is corrupted or not
def validate_video(vid_path,train_transforms):
      transform = train_transforms
      count = 20
      video_path = vid_path
      frames = []
      a = int(100/count)
      first_frame = np.random.randint(0,a)
      temp_video = video_path.split('/')[-1]
      for i,frame in enumerate(frame_extract(video_path)):
        frames.append(transform(frame))
        if(len(frames) == count):
          break
      frames = torch.stack(frames)
      frames = frames[:count]
      return frames
#extract a from from video
def frame_extract(path):
  vidObj = cv2.VideoCapture(path)
  success = 1
  while success:
      success, image = vidObj.read()
      if success:
          yield image

im_size = 112
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
                                        transforms.ToPILImage(),
                                        transforms.Resize((im_size,im_size)),
                                        transforms.ToTensor(),
                                        transforms.Normalize(mean,std)])

video_fil = glob.glob(r'D:\major project\dataset\face only\FF_Face_only_data\*.mp4')
video_fil += glob.glob(r'D:\major project\dataset\face only\DFDC_FAKE_Face_only_data\*.mp4')
video_fil += glob.glob(r'D:\major project\dataset\face only\Celeb_real_face_only\*.mp4')
video_fil += glob.glob(r'D:\major project\dataset\face only\Celeb_fake_face_only\*.mp4')
video_fil += glob.glob(r'D:\major project\dataset\face only\DFDC_REAL_Face_only_data\*.mp4')
print("Total no of videos :" , len(video_fil))
print(video_fil)
count = 0;
for i in video_fil:
  try:
    count+=1
    validate_video(i,train_transforms)
  except:
    print("Number of video processed: " , count ," Remaining : " , (len(video_fil) - count))
    print("Corrupted video is : " , i)
    continue
print((len(video_fil) - count))

Total no of videos : 4808
['D:\\major project\\dataset\\face only\\FF_Face_only_data\\819.mp4', 'D:\\major project\\dataset\\face only\\FF_Face_only_data\\821.mp4', 'D:\\major project\\dataset\\face only\\FF_Face_only_data\\821_812.mp4', 'D:\\major project\\dataset\\face only\\FF_Face_only_data\\823_584.mp4', 'D:\\major project\\dataset\\face only\\FF_Face_only_data\\824.mp4', 'D:\\major project\\dataset\\face only\\FF_Face_only_data\\824_419.mp4', 'D:\\major project\\dataset\\face only\\FF_Face_only_data\\825.mp4', 'D:\\major project\\dataset\\face only\\FF_Face_only_data\\825_074.mp4', 'D:\\major project\\dataset\\face only\\FF_Face_only_data\\826.mp4', 'D:\\major project\\dataset\\face only\\FF_Face_only_data\\826_833.mp4', 'D:\\major project\\dataset\\face only\\FF_Face_only_data\\827.mp4', 'D:\\major project\\dataset\\face only\\FF_Face_only_data\\827_817.mp4', 'D:\\major project\\dataset\\face only\\FF_Face_only_data\\830.mp4', 'D:\\major project\\dataset\\face only\\FF_Face_only

In [2]:
#to load preprocessod video to memory
import json
import glob
import numpy as np
import cv2
import copy
import random


video_files = glob.glob(r'D:\major project\dataset\face only\FF_Face_only_data\*.mp4')
video_files += glob.glob(r'D:\major project\dataset\face only\DFDC_FAKE_Face_only_data\*.mp4')
video_files += glob.glob(r'D:\major project\dataset\face only\Celeb_real_face_only\*.mp4')
video_files += glob.glob(r'D:\major project\dataset\face only\Celeb_fake_face_only\*.mp4')
video_files += glob.glob(r'D:\major project\dataset\face only\DFDC_REAL_Face_only_data\*.mp4')
random.shuffle(video_files)
random.shuffle(video_files)
frame_count = []
for video_file in video_files:
  cap = cv2.VideoCapture(video_file)
  if(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))<100):
    video_files.remove(video_file)
    continue
  frame_count.append(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("frames are " , frame_count)
print("Total no of video: " , len(frame_count))
print('Average frame per video:',np.mean(frame_count))

frames are  [148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 134, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 135, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 134, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 144, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 143, 148, 148, 142, 148, 148, 148, 148, 148, 148, 148, 148, 148, 143, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 148, 14

In [3]:
# load the video name and labels from csv
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Dataset
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import face_recognition
class video_dataset(Dataset):
    def __init__(self,video_names,labels,sequence_length = 60,transform = None):
        self.video_names = video_names
        self.labels = labels
        self.transform = transform
        self.count = sequence_length
    def __len__(self):
        return len(self.video_names)
    def __getitem__(self,idx):
        video_path = self.video_names[idx]
        frames = []
        a = int(100/self.count)
        first_frame = np.random.randint(0,a)
        temp_video = video_path.split('/')[-1]
        #print(temp_video)
        label = self.labels.iloc[(labels.loc[labels["file"] == temp_video].index.values[0]),1]
        if(label == 'FAKE'):
          label = 0
        if(label == 'REAL'):
          label = 1
        for i,frame in enumerate(self.frame_extract(video_path)):
          frames.append(self.transform(frame))
          if(len(frames) == self.count):
            break
        frames = torch.stack(frames)
        frames = frames[:self.count]
        #print("length:" , len(frames), "label",label)
        return frames,label
    def frame_extract(self,path):
      vidObj = cv2.VideoCapture(path) 
      success = 1
      while success:
          success, image = vidObj.read()
          if success:
              yield image
#plot the image
def im_plot(tensor):
    image = tensor.cpu().numpy().transpose(1,2,0)
    b,g,r = cv2.split(image)
    image = cv2.merge((r,g,b))
    image = image*[0.22803, 0.22145, 0.216989] +  [0.43216, 0.394666, 0.37645]
    image = image*255.0
    plt.imshow(image.astype(int))
    plt.show()
     


In [4]:
#count the number of fake and real videos
def number_of_real_and_fake_videos(data_list):
  header_list = ["file","label"]
  lab = pd.read_csv(r'D:\major project\Deepfake_detection_using_deep_learning-master\Model Creation\labels\Full_metadata.csv',names=header_list)
  fake = 0
  real = 0
  for i in data_list:
    temp_video = i.split('/')[-1]
    label = lab.iloc[(labels.loc[labels["file"] == temp_video].index.values[0]),1]
    if(label == 'FAKE'):
      fake+=1
    if(label == 'REAL'):
      real+=1
  return real,fake

In [5]:
import pandas as pd

def number_of_real_and_fake_videos(data_list):
    header_list = ["file","label"]
    lab = pd.read_csv(r'D:\major project\Deepfake_detection_using_deep_learning-master\Model Creation\labels\Gobal_metadata.csv', names=header_list)
    fake = 0
    real = 0
    for i in data_list:
        temp_video = i.split('/')[-1]
        label = lab.iloc[(lab.loc[lab["file"] == temp_video].index.values[0]), 1]
        if label == 'FAKE':
            fake += 1
        elif label == 'REAL':
            real += 1
    return real, fake


In [6]:
import pandas as pd

# Read the metadata file
metadata_path = r'D:\major project\Deepfake_detection_using_deep_learning-master\Model Creation\labels\Gobal_metadata.csv'
metadata_df = pd.read_csv(metadata_path, header=None, names=["file", "label"], index_col=False)

# Print the first few rows of the metadata DataFrame
print(metadata_df.head())

# Example of accessing individual entries in the DataFrame
print(metadata_df.iloc[0]["file"])  # Accessing the file name of the first entry
print(metadata_df.iloc[0]["label"])  # Accessing the label of the first entry


          file label
0      000.mp4  REAL
1  000_003.mp4  FAKE
2      001.mp4  REAL
3  001_870.mp4  FAKE
4      002.mp4  REAL
000.mp4
REAL


In [6]:
# load the labels and video in data loader
import random
import pandas as pd
from sklearn.model_selection import train_test_split

header_list = ["file","label"]
labels = pd.read_csv(r'D:\major project\Deepfake_detection_using_deep_learning-master\Model Creation\labels\Full_metadata.csv',names=header_list)
#print(labels)
train_videos = video_files[:int(0.8*len(video_files))]
valid_videos = video_files[int(0.8*len(video_files)):]
print("train : " , len(train_videos))
print("test : " , len(valid_videos))
# train_videos,valid_videos = train_test_split(data,test_size = 0.2)
# print(train_videos)

print("TRAIN: ", "Real:",number_of_real_and_fake_videos(train_videos)[0]," Fake:",number_of_real_and_fake_videos(train_videos)[1])
print("TEST: ", "Real:",number_of_real_and_fake_videos(valid_videos)[0]," Fake:",number_of_real_and_fake_videos(valid_videos)[1])


im_size = 112
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
                                        transforms.ToPILImage(),
                                        transforms.Resize((im_size,im_size)),
                                        transforms.ToTensor(),
                                        transforms.Normalize(mean,std)])

test_transforms = transforms.Compose([
                                        transforms.ToPILImage(),
                                        transforms.Resize((im_size,im_size)),
                                        transforms.ToTensor(),
                                        transforms.Normalize(mean,std)])
train_data = video_dataset(train_videos,labels,sequence_length = 10,transform = train_transforms)
#print(train_data)
val_data = video_dataset(valid_videos,labels,sequence_length = 10,transform = train_transforms)
train_loader = DataLoader(train_data,batch_size = 4,shuffle = True,num_workers = 4)
valid_loader = DataLoader(val_data,batch_size = 4,shuffle = True,num_workers = 4)
image,label = train_data[0]
im_plot(image[0,:,:,:])

train :  3829
test :  958


IndexError: index 0 is out of bounds for axis 0 with size 0

In [8]:
import random
import pandas as pd
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

# Assuming you have already defined the `number_of_real_and_fake_videos` function

class VideoDataset(Dataset):
    def __init__(self, video_files, labels_df, sequence_length, transform=None):
        self.video_files = video_files
        self.labels_df = labels_df
        self.sequence_length = sequence_length
        self.transform = transform

    def __len__(self):
        return len(self.video_files)

    def __getitem__(self, idx):
        video_file = self.video_files[idx]
        label = self.labels_df.loc[self.labels_df["file"] == video_file.split('/')[-1], "label"].values[0]

        # You need to define how to load your video data here
        # For example, you can load frames of the video and stack them into a tensor

        if self.transform:
            # Apply transformations if specified
            # Note: You need to define the transformation functions like ToPILImage, Resize, etc.
            #       before using them
            pass  # Add your transformations here

        return video_data, label  # Return your video data and label

# Load labels
header_list = ["file", "label"]
labels = pd.read_csv(r'D:\major project\Deepfake_detection_using_deep_learning-master\Model Creation\labels\Gobal_metadata.csv', names=header_list)

# Assuming you have already defined train_videos and valid_videos
# Split data into train and validation sets
train_videos, valid_videos = train_test_split(video_files, test_size=0.2, random_state=42)

# Print the number of train and test videos
print("Train:", len(train_videos))
print("Test:", len(valid_videos))

# Print the number of real and fake videos in train and test sets
print("TRAIN - Real:", number_of_real_and_fake_videos(train_videos)[0], " Fake:", number_of_real_and_fake_videos(train_videos)[1])
print("TEST - Real:", number_of_real_and_fake_videos(valid_videos)[0], " Fake:", number_of_real_and_fake_videos(valid_videos)[1])

# Define transformations
im_size = 112
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((im_size, im_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((im_size, im_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Create datasets and data loaders
train_data = VideoDataset(train_videos, labels, sequence_length=10, transform=train_transforms)
val_data = VideoDataset(valid_videos, labels, sequence_length=10, transform=test_transforms)

train_loader = DataLoader(train_data, batch_size=4, shuffle=True, num_workers=4)
valid_loader = DataLoader(val_data, batch_size=4, shuffle=True, num_workers=4)

# Note: You need to implement the `im_plot` function to visualize the video frames
#       or use any other visualization method you prefer.


Train: 3829
Test: 958


IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
import torch
from torch import nn
from torchvision import models

class EfficientNetLSTMModel(nn.Module):
    def __init__(self, num_classes, variant='b0', latent_dim=1280, lstm_layers=1, hidden_dim=1280, bidirectional=False):
        super(EfficientNetLSTMModel, self).__init__()

        # Load pre-trained EfficientNet variant
        self.efficientnet = models.efficientnet_b0(pretrained=True)  # Adjust variant as needed (b1, b2, etc.)

        # Freeze EfficientNet layers for fine-tuning
        for param in self.efficientnet.parameters():
            param.requires_grad = False

        # Extract features from EfficientNet
        self.feature_extractor = nn.Sequential(*list(self.efficientnet.children())[:-1])

        # Feature pooling (consider alternatives like global average pooling)
        self.avgpool = nn.AdaptiveAvgPool2d(1)

        # LSTM layers
        self.lstm = nn.LSTM(latent_dim, hidden_dim, lstm_layers, bidirectional=bidirectional)

        # Dropout for regularization
        self.dropout = nn.Dropout(0.4)

        # Classification layer
        self.linear = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        batch_size, seq_length, c, h, w = x.shape

        # Reshape input for efficient processing
        x = x.view(batch_size * seq_length, c, h, w)

        # Extract features using pre-trained EfficientNet
        fmaps = self.feature_extractor(x)

        # Feature pooling
        x = self.avgpool(fmaps)

        # Reshape for LSTM input
        x = x.view(batch_size, seq_length, -1)  # Adjust dimension based on pooling output

        # LSTM processing
        lstm_out, _ = self.lstm(x, None)

        # Final classification with dropout
        return fmaps, self.dropout(self.linear(torch.mean(lstm_out, dim=1)))


In [15]:
from torch import nn
from torchvision import models
import torch
import numpy as np

class MultiModelFaceGuard(nn.Module):
    def __init__(self, num_classes, lstm_hidden_dim=256, lstm_layers=2, attention_heads=8):
        super(MultiModelFaceGuard, self).__init__()
        
        # Teacher Model
        self.teacher_model = models.resnext50_32x4d(pretrained=True)
        self.teacher_model = nn.Sequential(*list(self.teacher_model.children())[:-2])
        self.teacher_lstm = nn.LSTM(2048, lstm_hidden_dim, lstm_layers, bidirectional=True)
        self.teacher_linear = nn.Linear(lstm_hidden_dim * 2, num_classes)
        self.teacher_avgpool = nn.AdaptiveAvgPool2d(1)
        
        # Student Model
        self.backbone = models.regnet_y_800mf(pretrained=True)
        backbone_features = self._get_backbone_features(self.backbone)
        self.backbone.trunk_output = nn.Identity()
        self.lstm = nn.LSTM(input_size=backbone_features, hidden_size=lstm_hidden_dim, num_layers=lstm_layers, batch_first=True)
        self.attention = nn.MultiheadAttention(lstm_hidden_dim, num_heads=attention_heads, batch_first=True)
        self.fc = nn.Linear(lstm_hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)

    def _get_backbone_features(self, backbone):
        with torch.no_grad():
            dummy_input = torch.rand(1, 3, 112, 112)
            backbone_features = backbone.trunk_output(backbone.stem(dummy_input)).shape[1]
        return backbone_features

    def forward(self, x):
        batch_size, seq_length, c, h, w = x.shape
        x = x.view(batch_size * seq_length, c, h, w)
        
        # Teacher Model Forward
        teacher_fmap = self.teacher_model(x)
        teacher_x = self.teacher_avgpool(teacher_fmap)
        teacher_x = teacher_x.view(batch_size, seq_length, 2048)
        teacher_x_lstm, _ = self.teacher_lstm(teacher_x)
        teacher_output = self.teacher_linear(torch.mean(teacher_x_lstm, dim=1))
        
        # Student Model Forward
        student_x = self.backbone(x)
        student_x = student_x.view(batch_size, seq_length, -1)
        student_x_lstm, _ = self.lstm(student_x)
        student_x, _ = self.attention(student_x_lstm, student_x_lstm, student_x_lstm)
        student_x = self.dropout(student_x)
        student_output = self.fc(student_x.mean(dim=1))
        
        return teacher_fmap, teacher_output, student_output



In [16]:
# Move the model to GPU
model = MultiModelFaceGuard(2).cuda()

# Create a dummy input tensor
dummy_input = torch.from_numpy(np.empty((1, 20, 3, 112, 112))).type(torch.cuda.FloatTensor)

# Forward pass through the model
teacher_fmap, teacher_output, student_output = model(dummy_input)

c:\Users\akema\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNeXt50_32X4D_Weights.IMAGENET1K_V1`. You can also use `weights=ResNeXt50_32X4D_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


RuntimeError: mat1 and mat2 shapes cannot be multiplied (20x32 and 784x1000)

In [17]:
#Model with feature visualization
from torch import nn
from torchvision import models
class Model(nn.Module):
    def __init__(self, num_classes,latent_dim= 2048, lstm_layers=1 , hidden_dim = 2048, bidirectional = False):
        super(Model, self).__init__()
        model = models.resnext50_32x4d(pretrained = True) #Residual Network CNN
        self.model = nn.Sequential(*list(model.children())[:-2])
        self.lstm = nn.LSTM(latent_dim,hidden_dim, lstm_layers,  bidirectional)
        self.relu = nn.LeakyReLU()
        self.dp = nn.Dropout(0.4)
        self.linear1 = nn.Linear(2048,num_classes)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
    def forward(self, x):
        batch_size,seq_length, c, h, w = x.shape
        x = x.view(batch_size * seq_length, c, h, w)
        fmap = self.model(x)
        x = self.avgpool(fmap)
        x = x.view(batch_size,seq_length,2048)
        x_lstm,_ = self.lstm(x,None)
        return fmap,self.dp(self.linear1(torch.mean(x_lstm,dim = 1)))

In [14]:
# Move the model to GPU
model = FaceGuardModel(2).cuda()

# Create a dummy input tensor
dummy_input = torch.from_numpy(np.empty((1, 20, 3, 112, 112))).type(torch.cuda.FloatTensor)

# Forward pass through the model
output = model(dummy_input)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (20x32 and 784x1000)

In [26]:
import torch
from torch import nn
from torchvision import models

class SelfAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(SelfAttention, self).__init__()
        self.hidden_dim = hidden_dim
        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(True),
            nn.Linear(64, 1)
        )

    def forward(self, encoder_outputs):
        energy = self.projection(encoder_outputs)
        weights = torch.softmax(energy.squeeze(-1), dim=1)
        outputs = (encoder_outputs * weights.unsqueeze(-1)).sum(dim=1)
        return outputs

class EnhancedModel(nn.Module):
    def __init__(self, num_classes, latent_dim=2048, lstm_layers=1, hidden_dim=2048, bidirectional=False):
        super(EnhancedModel, self).__init__()
        self.cnn_backbone = models.resnext50_32x4d(pretrained=True)
        self.lstm = nn.LSTM(latent_dim, hidden_dim, lstm_layers, bidirectional=bidirectional)
        self.attention = SelfAttention(hidden_dim)
        self.relu = nn.LeakyReLU()
        self.dropout = nn.Dropout(0.4)
        self.linear1 = nn.Linear(2048, num_classes)
        self.avgpool = nn.AdaptiveAvgPool2d(1)

        # Additional branches for feature visualization
        self.visualization_linear1 = nn.Linear(hidden_dim, 512)
        self.visualization_linear2 = nn.Linear(512, num_classes)

    def forward(self, x):
        batch_size, seq_length, c, h, w = x.shape
        x = x.view(batch_size * seq_length, c, h, w)
        fmap = self.cnn_backbone(x)
        x = self.avgpool(fmap)
        x = x.view(batch_size, seq_length, -1)
        x_lstm, _ = self.lstm(x)
        x_att = self.attention(x_lstm)
        x = self.dropout(x_att)

        # Additional branch for feature visualization
        x_visualization = self.visualization_linear1(x_lstm)
        x_visualization = self.relu(x_visualization)
        x_visualization = self.visualization_linear2(x_visualization)

        return fmap, self.linear1(x), x_visualization


    # Instantiate the enhanced model
  


In [30]:
enhanced_model = EnhancedModel(num_classes=2)

    # Create a dummy input tensor
dummy_input = torch.randn((1, 20, 3, 224, 224))

    # Forward pass through the enhanced model
fmap, output, visualization_output = enhanced_model(dummy_input)

In [24]:
model = ModelWithAttention(2).cuda()

    # Create a dummy input tensor
dummy_input = torch.randn((1, 20, 3, 224, 224)).cuda()
a, b = model(dummy_input)


    # Forward pass through the model


In [13]:
#Model with feature visualization
from torch import nn
from torchvision import models
class Model(nn.Module):
    def __init__(self, num_classes,latent_dim= 2048, lstm_layers=1 , hidden_dim = 2048, bidirectional = False):
        super(Model, self).__init__()
        model = models.resnext50_32x4d(pretrained = True) #Residual Network CNN
        self.model = nn.Sequential(*list(model.children())[:-2])
        self.lstm = nn.LSTM(latent_dim,hidden_dim, lstm_layers,  bidirectional)
        self.relu = nn.LeakyReLU()
        self.dp = nn.Dropout(0.4)
        self.linear1 = nn.Linear(2048,num_classes)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
    def forward(self, x):
        batch_size,seq_length, c, h, w = x.shape
        x = x.view(batch_size * seq_length, c, h, w)
        fmap = self.model(x)
        x = self.avgpool(fmap)
        x = x.view(batch_size,seq_length,2048)
        x_lstm,_ = self.lstm(x,None)
        return fmap,self.dp(self.linear1(torch.mean(x_lstm,dim = 1)))
     

     

In [14]:

model = Model(2).cuda()
a,b = model(torch.from_numpy(np.empty((1,20,3,112,112))).type(torch.cuda.FloatTensor))

c:\Users\akema\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\akema\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNeXt50_32X4D_Weights.IMAGENET1K_V1`. You can also use `weights=ResNeXt50_32X4D_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [15]:
import torch
from torch.autograd import Variable
import time
import os
import sys
import os
def train_epoch(epoch, num_epochs, data_loader, model, criterion, optimizer):
    model.train()
    losses = AverageMeter()
    accuracies = AverageMeter()
    t = []
    for i, (inputs, targets) in enumerate(data_loader):
        if torch.cuda.is_available():
            targets = targets.type(torch.cuda.LongTensor)
            inputs = inputs.cuda()
        _,outputs = model(inputs)
        loss  = criterion(outputs,targets.type(torch.cuda.LongTensor))
        acc = calculate_accuracy(outputs, targets.type(torch.cuda.LongTensor))
        losses.update(loss.item(), inputs.size(0))
        accuracies.update(acc, inputs.size(0))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        sys.stdout.write(
                "\r[Epoch %d/%d] [Batch %d / %d] [Loss: %f, Acc: %.2f%%]"
                % (
                    epoch,
                    num_epochs,
                    i,
                    len(data_loader),
                    losses.avg,
                    accuracies.avg))
    torch.save(model.state_dict(),'/content/checkpoint.pt')
    return losses.avg,accuracies.avg
def test(epoch,model, data_loader ,criterion):
    print('Testing')
    model.eval()
    losses = AverageMeter()
    accuracies = AverageMeter()
    pred = []
    true = []
    count = 0
    with torch.no_grad():
        for i, (inputs, targets) in enumerate(data_loader):
            if torch.cuda.is_available():
                targets = targets.cuda().type(torch.cuda.FloatTensor)
                inputs = inputs.cuda()
            _,outputs = model(inputs)
            loss = torch.mean(criterion(outputs, targets.type(torch.cuda.LongTensor)))
            acc = calculate_accuracy(outputs,targets.type(torch.cuda.LongTensor))
            _,p = torch.max(outputs,1) 
            true += (targets.type(torch.cuda.LongTensor)).detach().cpu().numpy().reshape(len(targets)).tolist()
            pred += p.detach().cpu().numpy().reshape(len(p)).tolist()
            losses.update(loss.item(), inputs.size(0))
            accuracies.update(acc, inputs.size(0))
            sys.stdout.write(
                    "\r[Batch %d / %d]  [Loss: %f, Acc: %.2f%%]"
                    % (
                        i,
                        len(data_loader),
                        losses.avg,
                        accuracies.avg
                        )
                    )
        print('\nAccuracy {}'.format(accuracies.avg))
    return true,pred,losses.avg,accuracies.avg
class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count
def calculate_accuracy(outputs, targets):
    batch_size = targets.size(0)

    _, pred = outputs.topk(1, 1, True)
    pred = pred.t()
    correct = pred.eq(targets.view(1, -1))
    n_correct_elems = correct.float().sum().item()
    return 100* n_correct_elems / batch_size

In [16]:
import seaborn as sn
#Output confusion matrix
def print_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    print('True positive = ', cm[0][0])
    print('False positive = ', cm[0][1])
    print('False negative = ', cm[1][0])
    print('True negative = ', cm[1][1])
    print('\n')
    df_cm = pd.DataFrame(cm, range(2), range(2))
    sn.set(font_scale=1.4) # for label size
    sn.heatmap(df_cm, annot=True, annot_kws={"size": 16}) # font size
    plt.ylabel('Actual label', size = 20)
    plt.xlabel('Predicted label', size = 20)
    plt.xticks(np.arange(2), ['Fake', 'Real'], size = 16)
    plt.yticks(np.arange(2), ['Fake', 'Real'], size = 16)
    plt.ylim([2, 0])
    plt.show()
    calculated_acc = (cm[0][0]+cm[1][1])/(cm[0][0]+cm[0][1]+cm[1][0]+ cm[1][1])
    print("Calculated Accuracy",calculated_acc*100)

In [17]:
def plot_loss(train_loss_avg,test_loss_avg,num_epochs):
  loss_train = train_loss_avg
  loss_val = test_loss_avg
  print(num_epochs)
  epochs = range(1,num_epochs+1)
  plt.plot(epochs, loss_train, 'g', label='Training loss')
  plt.plot(epochs, loss_val, 'b', label='validation loss')
  plt.title('Training and Validation loss')
  plt.xlabel('Epochs')
  plt.ylabel('Loss')
  plt.legend()
  plt.show()
def plot_accuracy(train_accuracy,test_accuracy,num_epochs):
  loss_train = train_accuracy
  loss_val = test_accuracy
  epochs = range(1,num_epochs+1)
  plt.plot(epochs, loss_train, 'g', label='Training accuracy')
  plt.plot(epochs, loss_val, 'b', label='validation accuracy')
  plt.title('Training and Validation accuracy')
  plt.xlabel('Epochs')
  plt.ylabel('Accuracy')
  plt.legend()
  plt.show()

In [21]:
from sklearn.metrics import confusion_matrix
#learning rate
lr = 1e-5#0.001
#number of epochs 
num_epochs = 20

optimizer = torch.optim.Adam(model.parameters(), lr= lr,weight_decay = 1e-5)

#class_weights = torch.from_numpy(np.asarray([1,15])).type(torch.FloatTensor).cuda()
#criterion = nn.CrossEntropyLoss(weight = class_weights).cuda()
criterion = nn.CrossEntropyLoss().cuda()
train_loss_avg =[]
train_accuracy = []
test_loss_avg = []
test_accuracy = []
for epoch in range(1,num_epochs+1):
    l, acc = train_epoch(epoch,num_epochs,train_loader,model,criterion,optimizer)
    train_loss_avg.append(l)
    train_accuracy.append(acc)
    true,pred,tl,t_acc = test(epoch,model,valid_loader,criterion)
    test_loss_avg.append(tl)
    test_accuracy.append(t_acc)
plot_loss(train_loss_avg,test_loss_avg,len(train_loss_avg))
plot_accuracy(train_accuracy,test_accuracy,len(train_accuracy))
print(confusion_matrix(true,pred))
print_confusion_matrix(true,pred)